In [7]:
import torch

torch.cuda.is_available()

True

## 核心硬件与环境信息

In [8]:
# 1. 查看可用 GPU 数量
print(f"可用 GPU 数量: {torch.cuda.device_count()}")

# 2. 查看当前使用的 GPU 设备
print(f"当前设备索引: {torch.cuda.current_device()}")
print(f"当前设备名称: {torch.cuda.get_device_name()}") # NVIDIA A100 80GB PCIe

# 3. 查看 CUDA 版本与 Pytorch 版本
print(f"PyTorch 版本: {torch.__version__}") # 2.4.0+cu118
print(f"CUDA 版本(PyTorch 绑定): {torch.version.cuda}") # 11.8
print(f"cuDNN 版本: {torch.backends.cudnn.version()}") # 90100

可用 GPU 数量: 8
当前设备索引: 0
当前设备名称: NVIDIA A100 80GB PCIe
PyTorch 版本: 2.4.0+cu118
CUDA 版本(PyTorch 绑定): 11.8
cuDNN 版本: 90100


## 显存与算力状态(训练前必看)

In [21]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')
if device.type == 'cuda':
    print(f'总显存: {torch.cuda.get_device_properties(device).total_memory / 1024**3:.2f} GB')
    print(f'已用显存: {torch.cuda.memory_allocated(device) / 1024**3:.2f} GB')
    print(f'空闲显存: {torch.cuda.memory_reserved(device) / 1024**3:.2f} GB')
    print(f'算力能力: {torch.cuda.get_device_capability(device)}') # 算力版本 (8, 0)，即对应 GPU 架构为 sm_80（A100）, 其支持 FP16/BF16 混合精度、TF32 等高效训练格式；



总显存: 79.15 GB
已用显存: 0.01 GB
空闲显存: 0.04 GB
算力能力: (8, 0)


In [17]:
import subprocess
import time
from IPython.display import clear_output, display

def watch_nvidia_smi(interval=30, max_iter=None):
    """
    在 Jupyter 中定时刷新 nvidia-smi 输出
    :param interval: 刷新间隔（秒），默认 30 秒
    :param max_iter: 最大刷新次数（None 表示无限循环）
    """
    iter_count = 0
    while True:
        # 清空当前单元格输出（模拟 watch 的刷新效果）
        clear_output(wait=True)
        
        # 调用 nvidia-smi 命令并获取输出
        result = subprocess.run(
            ["nvidia-smi"],  # 系统命令
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            encoding="utf-8"
        )
        
        # 显示输出结果
        print(f"===== nvidia-smi 实时监控 | 刷新次数: {iter_count+1} | 间隔 {interval} 秒 =====")
        print(result.stdout)
        
        # 检查终止条件
        iter_count += 1
        if max_iter and iter_count >= max_iter:
            break
        
        # 等待指定间隔
        time.sleep(interval)

# 启动监控（每30秒刷新，无限循环；按单元格停止按钮可终止）
watch_nvidia_smi(interval=30)

===== nvidia-smi 实时监控 | 刷新次数: 114 | 间隔 30 秒 =====
Tue Mar 10 15:40:24 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100 80GB PCIe          On  | 00000000:89:00.0 Off |                    0 |
| N/A   36C    P0              60W / 300W |   4356MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------

KeyboardInterrupt: 